In [1]:
from abc import ABC
from enum import Enum
from uuid import uuid4


class Element(Enum):
    WATER = "물"
    FIRE = "불"
    GRASS = "풀"

    def multiplier(self, target: "Element") -> float:
        if _COUNTERS[self] == target:
            return 2.0
        if _COUNTERS[target] == self:
            return 0.5
        return 1.0


_COUNTERS = {Element.WATER: Element.FIRE, Element.FIRE: Element.GRASS, Element.GRASS: Element.WATER}

In [2]:
# 능력치: HP, Level
class Attribute(ABC):
    def __init__(self, max_value, min_value=0):
        self.max_value = max_value
        self.min_value = min_value
        self.current = max_value

    def change_current(self, delta):
        self.current = max(self.min_value, min(self.current + delta, self.max_value))


class HP(Attribute):
    def __init__(self, max_hp):
        super().__init__(max_hp)
        self.base_hp = max_hp

    def damage(self, amount):
        self.change_current(-amount)

    def is_alive(self):
        return self.current > 0

    def level_up(self, level):
        self.max_value = self.base_hp + (level - 1) * 20


class Level:
    def __init__(self):
        self.level = 1
        self.exp = 0

    @property
    def max_exp(self):
        return self.level * 100

    def gain(self, amount) -> bool:
        """레벨 업을 하면 True 아니면 False"""
        self.exp += amount
        is_level_up = self.exp >= self.max_exp
        while self.exp >= self.max_exp:  # 경험치를 많이 먹으면 여러 번 레벨업
            self.exp -= self.max_exp
            self.level += 1
        return is_level_up

In [3]:
# 아이템, 무기, 인벤토리
class Item:
    def __init__(self, name):
        self.name = name
        self.uuid = uuid4()


class Weapon(Item):
    def __init__(self, name, base_power, element: Element):
        super().__init__(name)
        self.base_power = base_power
        self.element = element
        self.enhance = 0

    @property
    def power(self):
        return self.base_power + 5 * self.enhance

    def upgrade(self):
        self.enhance += 1

    def __str__(self):
        tag = f"+{self.enhance} " if self.enhance else ""
        return f"{tag}{self.name}[{self.element.value}]"


class Inventory:
    LIMIT = 1000

    def __init__(self):
        self.inventory = {}  # uuid -> Item

    @property
    def items(self):
        return list(self.inventory.values())

    def add(self, item):
        if len(self.inventory) >= self.LIMIT:
            return None
        self.inventory[item.uuid] = item
        return f"'{item.name}' 획득!"

    def find(self, name):
        return next((i for i in self.items if i.name == name), None)

    def remove(self, item):
        self.inventory.pop(item.uuid)

In [4]:
# 몬스터
class GameObject(ABC):
    def __init__(self, name):
        self.name = name


class Monster(GameObject):
    def __init__(self, name, max_hp, element, exp_reward, drop=None):
        super().__init__(name)
        self.hp = HP(max_hp)
        self.element = element
        self.exp_reward = exp_reward
        self.drop_item = [drop] if drop else []

    def damage(self, amount):
        self.hp.damage(amount)

    def death(self, causer) -> list[str]:
        logs = [f"{self.name} 처치! EXP +{self.exp_reward}"]
        logs += causer.gain_exp(self.exp_reward)
        for d in self.drop_item:
            if log := causer.inventory.add(d):
                logs.append(log)
        return logs


class Slime(Monster):
    def __init__(self):
        super().__init__("슬라임", 40, Element.GRASS, 60, Weapon("물의 창", 25, Element.WATER))


class Golem(Monster):
    def __init__(self):
        super().__init__("골렘", 120, Element.GRASS, 90, Item("강화석"))


class Dragon(Monster):
    def __init__(self):
        super().__init__("드래곤", 300, Element.FIRE, 500)  # 보스: 드랍 없음

In [5]:
# 공격, 캐릭터
class Attack:
    def __init__(self, causer, target):
        self.causer = causer
        self.target = target

    def execute(self) -> str:
        weapon = self.causer.weapon
        mult = weapon.element.multiplier(self.target.element)
        dmg = int(weapon.power * mult)  # 최종 대미지 = 강화 포함 공격력 * 속성 배율
        self.target.damage(dmg)
        return f"{self.causer.name}의 {weapon} 공격! {self.target.name}에게 {dmg} 피해 (x{mult})"


class Character(GameObject):
    def __init__(self, name, max_hp, weapon):
        super().__init__(name)
        self.hp = HP(max_hp)
        self.level = Level()
        self.weapon = weapon
        self.inventory = Inventory()

    def attack(self, enemy) -> list[str]:
        if not enemy.hp.is_alive():
            return []
        logs = [Attack(self, enemy).execute()]
        if not enemy.hp.is_alive():
            logs += enemy.death(self)
        return logs

    def gain_exp(self, exp) -> list[str]:
        if not self.level.gain(exp):
            return []
        self.hp.level_up(self.level.level)
        return [f"레벨 업! Lv.{self.level.level} (최대 HP {self.hp.max_value})"]

    def equip(self, name) -> bool:
        item = self.inventory.find(name)
        if not isinstance(item, Weapon):
            return False
        # 벗고 넣고 장착
        self.inventory.remove(item)
        self.inventory.add(self.weapon)
        self.weapon = item
        return True

    def upgrade_weapon(self) -> bool:
        stone = self.inventory.find("강화석")
        if stone is None:
            return False
        self.inventory.remove(stone)
        self.weapon.upgrade()
        return True


class Warrior(Character):
    def __init__(self, name):
        super().__init__(name, 150, Weapon("화염검", 30, Element.FIRE))

In [6]:
hero = Warrior("아서")

# 1) 슬라임 사냥
for line in hero.attack(Slime()):
    print(line)
print([i.name for i in hero.inventory.items])
print("---")

# 2) 골렘 사냥 (쓰러질 때까지 공격)
golem = Golem()
while golem.hp.is_alive():
    for line in hero.attack(golem):
        print(line)
print("---")

# 3) 보스전 준비: 무기 교체 + 강화
print(hero.equip("물의 창"), hero.weapon, [i.name for i in hero.inventory.items])
print(hero.upgrade_weapon(), hero.weapon, hero.weapon.power)
print("---")

# 4) 보스전
boss = Dragon()
turn = 0
while boss.hp.is_alive():
    turn += 1
    hero.attack(boss)
print(f"{turn}턴 만에 보스 처치! Lv.{hero.level.level}, EXP {hero.level.exp}")

아서의 화염검[불] 공격! 슬라임에게 60 피해 (x2.0)
슬라임 처치! EXP +60
'물의 창' 획득!
['물의 창']
---
아서의 화염검[불] 공격! 골렘에게 60 피해 (x2.0)
아서의 화염검[불] 공격! 골렘에게 60 피해 (x2.0)
골렘 처치! EXP +90
레벨 업! Lv.2 (최대 HP 170)
'강화석' 획득!
---
True 물의 창[물] ['강화석', '화염검']
True +1 물의 창[물] 30
---
5턴 만에 보스 처치! Lv.4, EXP 50
